# 04 — MAML meta-training pilot on CPU (corrected)

**What this notebook does.** It meta-trains the LSTM autoencoder with first-order MAML for
a short pilot run (500 outer steps), checks that training really updates the model, and
looks at what adaptation does on the meta-validation channels.

**Why.** It is a quick, cheap check that the training loop works before the long run in
notebook 06.

**Input.** The raw SMAP release. **Output.** `maml_pilot_seed42.pt`,
`maml_pilot_summary.json` and one figure.

### What the check of the original found

- **The uploaded `04_maml_training.ipynb` (learn2learn version) was correct.** It used
  `learn2learn.MAML(first_order=True)`, whose `clone()` keeps the link to the meta-model.
  A direct test gave gradients on all 20 parameter tensors, and the gradient matched the
  corrected loop of notebook 06 to a relative difference of 1e-7.
- **The Kaggle copy `04-maml-training (3).ipynb` was broken.** It deep-copied the model,
  adapted the copy, and called `.backward()` on the copy. The meta-model received no
  gradient (0 of 20 tensors) and never changed. The checkpoint that notebook 05 loaded
  (`maml_best_kaggle.pt`, step 5,500) came from this broken loop.

### What was corrected

1. The loop comes from `smap_common.py` (plain PyTorch, no learn2learn). The maths is the
   same first-order MAML. learn2learn is no longer maintained and does not install on
   current Python versions.
2. **Built-in checks:** after the first outer step every meta-parameter must have a
   non-zero gradient and the parameters must have moved; adaptation must move the weights;
   a flat validation curve raises a warning. The notebook also runs the broken Kaggle loop
   once to show that the checks catch it.
3. **No meta-test channel is used.** The original ran "sanity checks" on meta-test channels
   with their anomaly windows, used them to decide on more training, and diagnosed P-4 from
   them. That is selection with test labels. All checks here use meta-validation channels
   and normal data only.
4. **Validation is less noisy and no longer affects training.** The original drew one new
   random episode per validation channel at each check, from the same random stream as
   training. Validation now uses 4 fixed episodes per channel, drawn once.
5. **Inner steps set to 10**, as in notebooks 06–10. The original pilot used 5.
6. The checkpoint stores plain model weights (the original stored learn2learn's
   `module.`-prefixed keys). The final cell that called an undefined `load_lstm` was removed.

In [1]:
import os, sys

def _find_common():
    """Find smap_common.py: this folder when run locally, /kaggle/input on Kaggle."""
    for root in [os.getcwd(), "/kaggle/input"]:
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "smap_common.py" in files:
                    return d
    raise FileNotFoundError("smap_common.py not found. Run from the corrected_legacy folder, "
                            "or attach that folder to Kaggle as a Dataset.")

sys.path.insert(0, _find_common())
import smap_common as sc
SMOKE = os.environ.get("SMAP_SMOKE") == "1"     # tiny settings for testing only
OUT = sc.output_dir(smoke=SMOKE)
print("shared code:", sc.__file__)
print("outputs go to:", OUT)
print("code version:", sc.git_commit())

import copy, numpy as np, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, os.cpu_count() or 1))
CFG = dict(seed=42, n_outer=500, val_every=50, inner_lr=0.01, inner_steps=10, outer_lr=1e-3,
           tasks_per_batch=4, support_size=20, query_size=20, val_episodes_per_task=4,
           val_seed=2024, use_scheduler=False)
if SMOKE:
    CFG.update(n_outer=4, val_every=2, val_episodes_per_task=1)
print("device:", DEVICE, "| smoke test:" if SMOKE else "|", CFG)

shared code: /home/user/Objective-2/corrected_legacy/smap_common.py
outputs go to: /home/user/Objective-2/corrected_legacy/corrected_legacy_out
code version: 949e48b0f51a13289cdd327f73a3a76ac46d4ef5


device: cpu | {'seed': 42, 'n_outer': 500, 'val_every': 50, 'inner_lr': 0.01, 'inner_steps': 10, 'outer_lr': 0.001, 'tasks_per_batch': 4, 'support_size': 20, 'query_size': 20, 'val_episodes_per_task': 4, 'val_seed': 2024, 'use_scheduler': False}


## 1 — Data: meta-training and meta-validation channels only

In [2]:
data, _ = sc.build_smap_dataset(sc.META_TRAIN + sc.META_VAL)
train_windows = {c: data[c]["normal_windows"] for c in sc.META_TRAIN}
val_episodes = sc.fixed_episodes({c: data[c]["normal_windows"] for c in sc.META_VAL},
                                 CFG["val_episodes_per_task"], CFG["val_seed"],
                                 CFG["support_size"], CFG["query_size"])
print(f"meta-train channels {len(train_windows)} | meta-val channels {len(sc.META_VAL)} | "
      f"fixed validation episodes {len(val_episodes)}")

meta-train channels 39 | meta-val channels 6 | fixed validation episodes 24


## 2 — Check the training step before training

One outer step with the corrected loop, then one with the broken loop from the Kaggle
copy of this notebook, both starting from the same weights. The check should pass for the
first and fail for the second.

In [3]:
sc.seed_everything(0)
probe = sc.LSTMAutoencoder(25).to(DEVICE)
rng = np.random.RandomState(0)
eps = [tuple(sc.to_tensor(a, DEVICE) for a in sc.sample_episode(train_windows[c], rng))
       for c in sc.META_TRAIN[:4]]

good = copy.deepcopy(probe); opt = torch.optim.Adam(good.parameters(), lr=1e-3)
before = [p.detach().clone() for p in good.parameters()]
sc.fomaml_outer_step(good, opt, eps, 0.01, 10)
print("corrected loop:", sc.check_outer_step(good, before))

bad = copy.deepcopy(probe); opt = torch.optim.Adam(bad.parameters(), lr=1e-3)
before = [p.detach().clone() for p in bad.parameters()]
meta = 0
for s, q in eps:                         # the loop from '04-maml-training (3)'
    learner = sc.inner_adapt(bad, s, 0.01, 10)
    meta = meta + torch.nn.functional.mse_loss(learner(q), q)
opt.zero_grad(); (meta / len(eps)).backward(); opt.step()
try:
    sc.check_outer_step(bad, before)
    raise RuntimeError("the check did not catch the broken loop")
except AssertionError as e:
    print("broken loop caught:", e)

corrected loop: {'n_params_with_grad': 20, 'meta_param_change_l2': 0.22611100761275882}


broken loop caught: meta-parameters without gradient: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


## 3 — Meta-train (pilot)

In [4]:
sc.seed_everything(CFG["seed"])
model = sc.LSTMAutoencoder(25).to(DEVICE)
model, info = sc.train_maml(
    model, train_windows, val_episodes, DEVICE, seed=CFG["seed"], n_outer=CFG["n_outer"],
    val_every=CFG["val_every"], inner_lr=CFG["inner_lr"], inner_steps=CFG["inner_steps"],
    outer_lr=CFG["outer_lr"], tasks_per_batch=CFG["tasks_per_batch"],
    support_size=CFG["support_size"], query_size=CFG["query_size"],
    use_scheduler=CFG["use_scheduler"], ckpt_path=os.path.join(OUT, "maml_pilot_ckpt.pt"))
print("checks:", info["checks"])
print(f"best validation loss {info['best_val']:.6f} at step {info['best_step']} of {info['steps_reached']}")

step     50 | train 0.014726 | val 0.017796 | best 0.017796 @ 50 | 201s


step    100 | train 0.012059 | val 0.016354 | best 0.016354 @ 100 | 229s


step    150 | train 0.016879 | val 0.014622 | best 0.014622 @ 150 | 257s


step    200 | train 0.015220 | val 0.013984 | best 0.013984 @ 200 | 286s


step    250 | train 0.018425 | val 0.012823 | best 0.012823 @ 250 | 314s


step    300 | train 0.013478 | val 0.012193 | best 0.012193 @ 300 | 340s


step    350 | train 0.012728 | val 0.010911 | best 0.010911 @ 350 | 368s


step    400 | train 0.011322 | val 0.010353 | best 0.010353 @ 400 | 397s


step    450 | train 0.008391 | val 0.009981 | best 0.009981 @ 450 | 427s


step    500 | train 0.012488 | val 0.009671 | best 0.009671 @ 500 | 454s
checks: {'outer_step': {'n_params_with_grad': 20, 'meta_param_change_l2': 0.22566946479172975}, 'adaptation_at_step_1': {'relative_param_change': 0.00014713999878435735, 'support_loss_before': 0.00823917891830206, 'support_loss_after': 0.008138847537338734}, 'val_loss_flat': False}
best validation loss 0.009671 at step 500 of 500


## 4 — Training curves

In [5]:
h = info["history"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([r["step"] for r in h], [r["train_loss"] for r in h], marker=".", label="training batch loss")
ax.plot([r["step"] for r in h], [r["val_loss"] for r in h], marker="o", label="validation loss (fixed episodes)")
ax.axvline(info["best_step"], ls="--", color="green", label=f"best step {info['best_step']}")
ax.set_xlabel("outer step"); ax.set_ylabel("query MSE after adaptation"); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(OUT, "04_maml_pilot_curves.png"), dpi=120); plt.close(fig)

## 5 — What adaptation does (meta-validation channels, normal data only)

For each validation episode we adapt the pilot model with 0 and with 10 inner steps and
compare the query loss. We also record how far the weights move. If 10 steps barely move
the weights or barely change the loss, adaptation is close to inert.

In [6]:
rows = []
for ch, s, q in val_episodes:
    st, qt = sc.to_tensor(s, DEVICE), sc.to_tensor(q, DEVICE)
    a = sc.inner_adapt(model, st, CFG["inner_lr"], CFG["inner_steps"])
    rep = sc.adaptation_report(model, a, st)
    with torch.no_grad():
        q0 = torch.nn.functional.mse_loss(model(qt), qt).item()
        q10 = torch.nn.functional.mse_loss(a(qt), qt).item()
    rows.append({"channel": ch, "query_loss_0_steps": q0, "query_loss_10_steps": q10, **rep})
import pandas as pd
adapt = pd.DataFrame(rows).groupby("channel").mean()
print(adapt.round(6).to_string())

         query_loss_0_steps  query_loss_10_steps  relative_param_change  support_loss_before  support_loss_after
channel                                                                                                         
A-8                0.003585             0.003576               0.000040             0.002708            0.002698
D-11               0.001959             0.001949               0.000062             0.002604            0.002579
D-5                0.014806             0.014481               0.000274             0.010967            0.010458
E-12               0.011840             0.011317               0.000291             0.013332            0.012760
E-7                0.013225             0.012791               0.000282             0.012444            0.011858
G-1                0.014236             0.013914               0.000235             0.012156            0.011803


## 6 — Save

In [7]:
torch.save({"model_state_dict": model.state_dict(), "step": info["best_step"], "val_loss": info["best_val"],
            "config": CFG, "git_commit": sc.git_commit()}, os.path.join(OUT, "maml_pilot_seed42.pt"))
print(sc.save_json(os.path.join(OUT, "maml_pilot_summary.json"),
                   {"kind": "pilot run on meta-train/meta-val channels only; no test data used",
                    "smoke_test": SMOKE, "config": CFG, "training": info,
                    "adaptation_on_meta_val": adapt.reset_index().to_dict(orient="records")}))

/home/user/Objective-2/corrected_legacy/corrected_legacy_out/maml_pilot_summary.json
